In [ ]:
# # !pip install numpy==1.23.1
# ## numpy 변경 절대 금지!!
# import numpy as np
# print(np.__version__)
# !python --version

2.0.2
Python 3.11.13


In [ ]:
!pip install mxnet
!pip install gluonnlp==0.8.0
!pip install pandas tqdm
!pip install sentencepiece
!pip install transformers
!pip install torch

  Using cached gluonnlp-0.8.0.tar.gz (235 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for gluonnlp: filename=gluonnlp-0.8.0-py3-none-any.whl size=292696 sha256=3e3a2c2bb709990178b28cc2e3c9d419a3b10fc92e35c8c70e91afa00cdf82c5
  Stored in directory: /root/.cache/pip/wheels/d4/33/10/735728ff9ab4891a97789d842c70354bdb368715e35f18fb9e
Successfully built gluonnlp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB

In [6]:
!pip install wandb -qU

In [ ]:
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'
!pip install git+https://github.com/huggingface/trl.git

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-qhzrvv9x/kobert-tokenizer_2d4b279d783c4b3280528e4630b10438
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-qhzrvv9x/kobert-tokenizer_2d4b279d783c4b3280528e4630b10438
  Resolved https://github.com/SKTBrain/KoBERT.git to commit fcd729f2f4b37858f333597c0782388ada51eb5f
  Preparing metadata (setup.py) ... done
  Created wheel for kobert_tokenizer: filename=kobert_tokenizer-0.1-py3-none-any.whl size=4633 sha256=6f9c4eb327ea5ce40b6c790874449231fc215c148e4858b60647409fbd62bb51
  Stored in directory: /tmp/pip-ephem-wheel-cache-bnh8_fi5/wheels/b7/95/7c/3f38b039ae9422cc4a51702b37e352d0585442cbe60285f1f2
Successfully built kobert_tokenizer
  Cloning https://github.com/huggingface/trl.git to /tmp/pip-req-build-2ldovgli
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-2ldovgli
  Resolved https://github.com

In [ ]:
import numpy as np ## 버전 오류로 인한 에러 해결
np.bool = np.bool_

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gluonnlp as nlp
from torch.utils.data import DataLoader, Dataset

# from transformers import AutoTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from tqdm.notebook import tqdm

ModuleNotFoundError: No module named 'gluonnlp'

In [10]:
from google.colab import userdata
from huggingface_hub import login, upload_file
import wandb

NAME = 'fact_opinion/'

# Hugging Face Access Token 및 리포지토리 이름 설정
hf_token = userdata.get('HF_TOKEN')
wandb_key = userdata.get('WANDB_KEY')
path = userdata.get('PATH')
repo_name = "Dang-gu/oss_fact_opinion"

# Hugging Face 로그인
login(token=hf_token)

wandb.login(key=wandb_key)
wandb.init(project='fact_tory', name='fact_opinion_finetune')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: danggu-su (codi_dankook) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# GPU를 사용할 것이므로, CUDA 세팅
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel, AutoModelForSequenceClassification

# KoBERT 토크나이저 & 모델
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
bertmodel = BertModel.from_pretrained('skt/kobert-base-v1', return_dict=False)

model = AutoModelForSequenceClassification.from_pretrained(
    "skt/kobert-base-v1",
    num_labels=2     # 🔁 클래스 수 꼭 맞춰줘 (negative/neutral/positive)
)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Danggu/Dataset/fact_opinion.csv')
data = data[(data['label'] == '사실형')|(data['label'] == '예측형')]

In [ ]:
## 클래스 불균형 문제 있음

from sklearn.utils import resample

# 사실형과 예측형 분리
data_fact = data[data['label'] == '사실형']
data_pred = data[data['label'] == '예측형']

# 사실형을 예측형 수만큼 랜덤 샘플링
data_fact_downsampled = resample(data_fact,
                                 replace=False,
                                 n_samples=len(data_pred),
                                 random_state=42)

balanced_data = pd.concat([data_fact_downsampled, data_pred])

print(balanced_data)

                                                    text label
1256    무엇보다 우선한다는 안보를 위하여 국가의 생산성은 경쟁국의 절반 이하로 저하된 것이다.   사실형
12883     기준대로라면 거의 대부분 중소형 조선소들이 철퇴를 맞을 가능성이 높다고 보고 있다.   사실형
49033  노벨 문학상을 받은 거장 사뮈엘 베케트(아일랜드)의 희곡 ＇고도를 기다리며＇는 끝없...   사실형
39672  이같은 내용은 2020년까지 흡연율을 20%로 낮추려는 복지부의 금연 종합대책에 포...   사실형
57009  이와 관련해 어 차관은 '개별 기업이 소송 등 법적 행동에 나설 수 있겠지만 정부로...   사실형
...                                                  ...   ...
64597  블리자드 및 디아블로와 관련한 특별 영상을 통해 ‘디아블로 이모탈’의 각종 정보를 ...   예측형
64599         버거킹과 협력해 제휴 상품을 출시하고 추가 제휴 프로모션도 진행할 예정이다.   예측형
64601         블리자드는 출시 이후 직업변경 기능도 선보여 이용자 경험을 확대할 예정이다.   예측형
64604   또 새로운 스토리와 지역, 던전, 신규 보스를 지속 추가하며 콘텐츠를 늘려갈 예정이다.   예측형
64746  금융당국부터 정부가 영향력을 행사하는 국책 금융기관의 수장들에 대한 대규모 인사가 ...   예측형

[6032 rows x 2 columns]


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# 1. 라벨 숫자화
label_map = {"사실형": 0, "예측형": 1}
balanced_data["label"] = balanced_data["label"].map(label_map)

# 2. Dataset 객체로 변환 + train/test 분리
dataset = Dataset.from_dict(balanced_data).train_test_split(test_size=0.2)

# 3. tokenizer 준비
tokenizer = AutoTokenizer.from_pretrained("skt/kobert-base-v1")

# 4. 토크나이저 함수 정의
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

# 5. 데이터셋에 tokenizer 적용
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4825 [00:00<?, ? examples/s]

Map:   0%|          | 0/1207 [00:00<?, ? examples/s]

In [ ]:
# 6. torch tensor로 변환
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
tokenized_dataset['train']

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4825
})

In [ ]:
#evalution compute_metrics 선언

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir= userdata.get("OUTPUT_DIR"),          # output directory
    report_to='wandb',                          # wandb 사용
    num_train_epochs=20,                        # 학습 epoch 설정
    per_device_train_batch_size=32,             # train batch_size 설정
    per_device_eval_batch_size=32,
    logging_dir= userdata.get("LOGGING_DIR"),            # directory for storing logs
    logging_steps=100,                          # 학습 log 기록 단위
    save_total_limit=2,
    eval_strategy="epoch",   # ← eval_strategy 오타 수정 (eval → evaluation)
    save_strategy="epoch",
    load_best_model_at_end=True
)

# 최적화 알고리즘(optimizer) 설정
optimizer = AdamW(model.parameters(), lr=2e-5)


#initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    optimizers=(optimizer, None),
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

# 모델 학습
trainer.train()
print(trainer.state.best_model_checkpoint)

/tmp/ipython-input-54-1254311395.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.689500,0.628376,0.652858,0.665603,0.644513,0.688119
2,0.638900,0.588259,0.698426,0.703583,0.694534,0.712871
3,0.608100,0.563603,0.723281,0.704947,0.758555,0.658416
4,0.585400,0.550003,0.715824,0.727562,0.701378,0.755776
5,0.565700,0.557857,0.725766,0.744402,0.699565,0.795380
6,0.546800,0.550222,0.730737,0.719586,0.754069,0.688119
7,0.522000,0.540761,0.739851,0.734797,0.752595,0.717822
8,0.506800,0.554095,0.721624,0.696203,0.770000,0.635314
9,0.494100,0.581554,0.734051,0.728196,0.747826,0.709571
10,0.447600,0.551678,0.746479,0.756757,0.730061,0.785479


/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Model/Fact_Opnion/checkpoint-1057


In [ ]:
trainer.save_model(userdata.get("BEST_MODEL"))
tokenizer.save_pretrained(userdata.get("BEST_MODEL"))

('/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Model/Fact_Opnion/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Model/Fact_Opnion/best_model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Model/Fact_Opnion/best_model/spiece.model',
 '/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Model/Fact_Opnion/best_model/added_tokens.json',
 '/content/drive/MyDrive/Colab_Notebooks/OSS_AI/Model/Fact_Opnion/best_model/tokenizer.json')

In [ ]:
# 테스트(Test) 데이터셋 평가
trainer.evaluate(tokenized_dataset['test'])

{'eval_loss': 0.5407605767250061,
 'eval_accuracy': 0.7398508699254349,
 'eval_f1': 0.7347972972972973,
 'eval_precision': 0.7525951557093425,
 'eval_recall': 0.7178217821782178,
 'eval_runtime': 7.8415,
 'eval_samples_per_second': 153.924,
 'eval_steps_per_second': 4.846,
 'epoch': 20.0}

In [ ]:
# from transformers import BertTokenizer, BertForSequenceClassification
# model = AutoModelForSequenceClassification.from_pretrained(
#     model_checkpoint,
#     num_labels=2     # 🔁 클래스 수 동일하게 유지
# )

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = userdata.get("BEST_MODEL")

# 모델과 토크나이저 로드
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

username = "Dang-gu"
MODEL_NAME = 'oss_fact_opinion_finetuning'

api.create_repo(
    token=hf_token,
    repo_id=f"{username}/{MODEL_NAME}",
    repo_type="model"
)

RepoUrl('https://huggingface.co/Dang-gu/oss_fact_opinion_finetuning', endpoint='https://huggingface.co', repo_type='model', repo_id='Dang-gu/oss_fact_opinion_finetuning')

In [ ]:
from huggingface_hub import HfApi, HfFolder, upload_folder,create_repo
# 사용자 정보
username = "Dang-gu"  # 🤚 너의 HF 계정명 (주의: 실제로는 소문자/하이픈 허용)
model_name = "oss_fact_opinion_finetuning"
model_dir = userdata.get("BEST_MODEL")

# 전체 경로
repo_id = 'Dang-gu/fact_opinion'

# HF API 객체 생성
api = HfApi()

create_repo(repo_id, repo_type="model")

# 🔧 모델 업로드
upload_folder(
    folder_path=model_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="upload trained KoBERT fact/opinion classifier",
    ignore_patterns=[".ipynb_checkpoints"]
)

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Dang-gu/fact_opinion/commit/1243fcd3bfd3e1d4e8911b24cbcc3d27f16a9c81', commit_message='upload trained KoBERT fact/opinion classifier', commit_description='', oid='1243fcd3bfd3e1d4e8911b24cbcc3d27f16a9c81', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Dang-gu/fact_opinion', endpoint='https://huggingface.co', repo_type='model', repo_id='Dang-gu/fact_opinion'), pr_revision=None, pr_num=None)

In [ ]:
## 실제 기사로 판단